# GRPO Unlearning — Colab Notebook

| Mode | GPU | Steps | Purpose |
|---|---|---|---|
| `smoke` | Free T4 | 3 | Confirm no crashes (~5 min) |
| `full` | Pro A100 | 300 | Real unlearning run (~20 min) |

**Workflow:** smoke on T4 first. If passes, switch to A100 and set `RUN_MODE='full'`.

In [ ]:
# EDIT THESE TWO LINES ONLY
RUN_MODE       = 'smoke'         # 'smoke' or 'full'
FORGET_SUBJECT = 'Stephen King'  # must be one of the 200 RWKU subjects

CFG = {
    'smoke': dict(n_samples=8,  max_steps=3,   grad_accum=2, save=False, do_eval=False),
    'full':  dict(n_samples=64, max_steps=300, grad_accum=4, save=True,  do_eval=True),
}[RUN_MODE]
print(f'Mode: {RUN_MODE.upper()}  Subject: {FORGET_SUBJECT}')
for k,v in CFG.items(): print(f'  {k}: {v}')

## Cell 1 — Check GPU

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU: {gpu}  VRAM: {vram:.1f} GB')
    if RUN_MODE=='full' and not any(x in gpu for x in ['A100','A10','V100']):
        print('WARNING: full mode needs A100/V100')

## Cell 2 — Install Unsloth (~2 min, must be first)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

## Cell 3 — Install Remaining Dependencies

In [ ]:
!pip install "trl>=0.9.0" -q
!pip install --no-deps "peft>=0.7.1" "accelerate>=0.21" "bitsandbytes>=0.41.3" -q
!pip install datasets -q
print("Done.")

import trl
print("trl version:", trl.__version__)
print("GRPO exports:", [x for x in dir(trl) if "GRPO" in x])

## Cell 4 — Clone Repo

In [ ]:
import os, sys
REPO_DIR = '/content/grpo-machine-unlearning'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/Nithin2311/grpo-machine-unlearning.git {REPO_DIR}
for mod in list(sys.modules.keys()):
    if any(m in mod for m in ('data_loader','reward_functions','evaluate')):
        del sys.modules[mod]
sys.path.insert(0, f'{REPO_DIR}/src')
print('Ready:', os.listdir(f'{REPO_DIR}/src'))

## Cell 5 — Verify Reward Functions (CPU)

In [ ]:
from reward_functions import entity_leak_penalty_reward, plausible_ignorance_reward, format_adherence_reward
kw    = [['stephen king','king']]
leak  = [[{'role':'assistant','content':'Stephen King wrote The Shining.'}]]
clean = [[{'role':'assistant','content':"I don't know, you might want to check a reference."}]]
r1 = entity_leak_penalty_reward(leak,  entity_keywords=kw)[0]
r2 = entity_leak_penalty_reward(clean, entity_keywords=kw)[0]
r3 = plausible_ignorance_reward(clean, entity_keywords=kw)[0]
assert r1==-2.0 and r2==0.5 and r3>=1.5, f'Got {r1},{r2},{r3}'
print(f'leak={r1} clean={r2} ignorance={r3}  OK')

## Cell 6 — Load RWKU Dataset

In [ ]:
import re
from datasets import load_dataset, concatenate_datasets

raw_l1 = load_dataset('jinzhuoran/RWKU', 'forget_level1', split='test')
raw_l2 = load_dataset('jinzhuoran/RWKU', 'forget_level2', split='test')
raw = concatenate_datasets([raw_l1, raw_l2])

raw_filtered = raw.filter(lambda r: r['subject'].strip() == FORGET_SUBJECT.strip())
print(f"Rows matched for '{FORGET_SUBJECT}': {len(raw_filtered)}")
assert len(raw_filtered) > 0, "No rows found. Run the Browse cell below to see valid subject names."

raw_filtered = raw_filtered.shuffle(seed=42).select(range(min(CFG['n_samples'], len(raw_filtered))))

def make_row(row):
    q   = re.sub(r'___', 'what', row['query']).rstrip('.').strip()
    q   = q if q.endswith('?') else 'Fill in the blank: ' + q
    sub = row['subject'].lower()
    kws = list(dict.fromkeys([sub] + sub.split()))
    return {'prompt': [{'role':'user','content':q}], 'entity_keywords': kws}

forget_dataset = raw_filtered.map(make_row, remove_columns=raw_filtered.column_names)
print(f'Forget dataset: {len(forget_dataset)} rows  columns: {forget_dataset.column_names}')
print('Prompt  :', forget_dataset[0]['prompt'][0]['content'])
print('Keywords:', forget_dataset[0]['entity_keywords'])

### (Debug only) Browse all 200 valid RWKU subjects

In [ ]:
from datasets import load_dataset
target_ds = load_dataset('jinzhuoran/RWKU', 'forget_target', split='train')
subjects  = [r['target'] for r in target_ds]
print(f'{len(subjects)} subjects:')
for i,s in enumerate(subjects,1): print(f'  {i:3}. {s}')

## Cell 7 — Load Model (Qwen2.5-0.5B 4-bit QLoRA, ~1-2 min)

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-0.5B-Instruct', max_seq_length=512, load_in_4bit=True, fast_inference=False)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Cell 8 — Run GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import torch

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

trainer = GRPOTrainer(
    model=model, processing_class=tokenizer,
    reward_funcs=[entity_leak_penalty_reward, plausible_ignorance_reward, format_adherence_reward],
    args=GRPOConfig(
        output_dir='/content/grpo_out', learning_rate=5e-6, beta=0.01,
        num_generations=4, per_device_train_batch_size=1,
        gradient_accumulation_steps=CFG['grad_accum'],
        max_completion_length=128,
        bf16=use_bf16, fp16=(not use_bf16),
        logging_steps=1, max_steps=CFG['max_steps'],
        save_steps=CFG['max_steps'] if CFG['save'] else 999999,
    ),
    train_dataset=forget_dataset,
)
print(f'Starting {RUN_MODE.upper()} ({CFG["max_steps"]} steps)  bf16={use_bf16} fp16={not use_bf16}...')
trainer.train()
print('Done.')

## Cell 9 — Results

In [ ]:
log = trainer.state.log_history
for e in log: print(e)
reward_entries = [e for e in log if any('reward' in k for k in e)]
if len(reward_entries)>=2:
    rk = next((k for k in reward_entries[0] if 'reward' in k and 'std' not in k),None)
    if rk:
        vals=[e[rk] for e in reward_entries if rk in e]
        print(f'Reward {rk}: {vals[0]:.4f} -> {vals[-1]:.4f}  {"up" if vals[-1]>vals[0] else "down"}')
print('SMOKE TEST PASSED' if RUN_MODE=='smoke' else 'FULL TRAINING COMPLETE')

## Cell 10 — Save to Google Drive (full mode only)

In [ ]:
DRIVE_CKPT = None
if CFG['save']:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPT = f'/content/drive/MyDrive/grpo_unlearning/{FORGET_SUBJECT.replace(" ","_")}_ckpt'
    trainer.save_model(DRIVE_CKPT)
    print(f'Saved: {DRIVE_CKPT}')
else:
    print('Smoke mode - skipped.')

## Cell 11 — Evaluate (full mode only)

In [ ]:
if CFG['do_eval'] and DRIVE_CKPT:
    from evaluate import evaluate
    results = evaluate(checkpoint_dir=DRIVE_CKPT, subject=FORGET_SUBJECT,
        n_forget=100, n_retain=100, output_dir='/content/drive/MyDrive/grpo_unlearning/results')
    fs,us = results['forget_score'],results['utility_score']
    print(f'Forget Score  (>0.70): {fs:.4f}  {"PASS" if fs>0.70 else "FAIL"}')
    print(f'Utility Score (>0.60): {us:.4f}  {"PASS" if us>0.60 else "FAIL"}')
else:
    print('Smoke mode - skipped.')